# Llama 3.1 8B Calibrated Listwise Ranking
Combined Logit Margin + Entropy confidence with temperature scaling.

In [1]:
# CELL 1: Install dependencies
!pip install -q accelerate peft
!pip install -q --upgrade transformers
!pip install -U bitsandbytes>=0.46.1
print('Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 86.9 MB/s eta 0:00:00
Done!


In [2]:
# CELL 2: Mount Drive and set paths

from google.colab import drive
drive.mount('/content/drive')

MODEL_NAME = 'Llama3.1-8B'  #Qwen2.5-7B, Llama3.1-8B
BASE_PATH    = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/Dataset/'
RESULTS_PATH = f'/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_ Calibrated and Adaptive Listwise Listwise Ranking/{MODEL_NAME}/'

import os
os.makedirs(RESULTS_PATH, exist_ok=True)

Mounted at /content/drive


In [3]:
# CELL 3: Imports and config
import pandas as pd
import numpy as np
import torch
import os, re, random, subprocess, sys

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import snapshot_download
from scipy.optimize import minimize_scalar
from scipy.special import expit
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
HF_TOKEN = ''

MODEL_IDS = {
    'Mistral-7B':  'mistralai/Mistral-7B-Instruct-v0.3',
    'Llama3.1-8B': 'meta-llama/Llama-3.1-8B-Instruct',
    'Qwen2.5-7B':  'Qwen/Qwen2.5-7B-Instruct',
}

ADAPTER_HF_IDS = {
    'Mistral-7B':  'NajatAlsa/mistral-scheme-a',
    'Llama3.1-8B': 'NajatAlsa/llama-scheme-a',
    'Qwen2.5-7B':  'NajatAlsa/qwen-scheme-a',
}

HF_MODEL_ID   = MODEL_IDS[MODEL_NAME]
ADAPTER_HF_ID = ADAPTER_HF_IDS[MODEL_NAME]
ADAPTER_LOCAL = f'/tmp/calib_adapter_{MODEL_NAME}'
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f'Model:  {MODEL_NAME}')
print(f'Device: {DEVICE}')

Model:  Llama3.1-8B
Device: cuda


In [4]:
# CELL 4: Load model and tokenizer
from huggingface_hub import snapshot_download

sys.path.append('/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/')
from prompt_config import SYSTEM_INSTRUCTION, build_user_content, build_assistant_output

# Download adapter from HuggingFace
ADAPTER_LOCAL = snapshot_download(repo_id=ADAPTER_HF_ID, token=HF_TOKEN)
print(f'Adapter downloaded to {ADAPTER_LOCAL}')

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
base_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    attn_implementation='eager'
)
# Load adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_LOCAL)
model.eval()
print('Model + adapter loaded successfully')

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Adapter downloaded to /root/.cache/huggingface/hub/models--NajatAlsa--llama-scheme-a/snapshots/70173300e8c190a7aca0a64e6f93abe83517fc09


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model + adapter loaded successfully


In [5]:
# CELL 5: Load datasets
val_df  = pd.read_csv(BASE_PATH + 'Scheme_A/obj3_val_A.csv')
test_df = pd.read_csv(BASE_PATH + 'Scheme_A/obj3_test_A.csv')

val_list_ids  = val_df['list_id'].unique().tolist()
test_list_ids = test_df['list_id'].unique().tolist()

print(f'Val lists:  {len(val_list_ids)}')
print(f'Test lists: {len(test_list_ids)}')

Val lists:  251
Test lists: 251


In [6]:
# CELL 6: Prompt builder — uses prompt_config.py (same as SFT evaluation)
def build_prompt(group_df):
    messages = [
        {'role': 'system', 'content': SYSTEM_INSTRUCTION},
        {'role': 'user',   'content': build_user_content(group_df)}
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

print('Prompt builder defined.')

Prompt builder defined.


In [7]:
# CELL 7: Inference with logprobs for confidence computation
def run_inference_with_logprobs(group_df, max_new_tokens=200):
    group = group_df.sample(frac=1, random_state=int(group_df['list_id'].iloc[0])).reset_index(drop=True)
    prompt    = build_prompt(group)
    inputs    = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids  = outputs.sequences[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    scores         = outputs.scores  # tuple of tensors, one per generated token

    return generated_text, scores, group


def compute_confidence(scores, group_df):
    display_ids = group_df['display_id'].tolist()

    # Get token IDs for each node's number suffix (e.g. "5" from "Node_5")
    node_token_ids = []
    for did in display_ids:
        num  = did.split('_')[1]
        tids = tokenizer.encode(num, add_special_tokens=False)
        if tids:
            node_token_ids.append(tids[0])

    margins   = []
    entropies = []

    for step_scores in scores:
        probs = torch.softmax(step_scores[0], dim=-1)

        # Node probs only — renormalised over candidate nodes
        node_probs = probs[node_token_ids]
        node_probs = node_probs / (node_probs.sum() + 1e-10)

        if len(node_probs) >= 2:
            sorted_p = torch.sort(node_probs, descending=True).values

            # Logit margin at node positions only
            margins.append((sorted_p[0] - sorted_p[1]).item())

            # Normalised entropy over node tokens only
            log_p       = torch.log(node_probs + 1e-10)
            entropy     = -torch.sum(node_probs * log_p).item()
            max_entropy = np.log(len(node_probs))
            entropies.append(1.0 - entropy / max_entropy)  # higher = more confident

    margin_score  = float(np.mean(margins))   if margins   else 0.0
    entropy_score = float(np.mean(entropies)) if entropies else 0.0

    print(f'  margin={margin_score:.4f}  entropy={entropy_score:.4f}  conf={np.sqrt(margin_score * entropy_score):.4f}')

    # Geometric mean of both signals
    confidence = float(np.sqrt(margin_score * entropy_score)) if (margin_score > 0 and entropy_score > 0) else 0.0
    return confidence


def parse_ranking(generated_text, group_df):
    display_ids = group_df['display_id'].tolist()
    lines       = generated_text.strip().split('\n')
    pred_ranks  = {}
    rank        = 1
    for line in lines:
        line = line.strip()
        for did in display_ids:
            if re.search(r'\b' + re.escape(did) + r'\b', line) and did not in pred_ranks:
                pred_ranks[did] = rank
                rank += 1
                break
    for did in display_ids:
        if did not in pred_ranks:
            pred_ranks[did] = rank
            rank += 1
    return pred_ranks


def compute_top1(group_df, pred_ranks):
    true_top1 = group_df[group_df['list_rank'] == 1]['display_id'].values[0]
    pred_top1 = min(pred_ranks, key=pred_ranks.get)
    return int(true_top1 == pred_top1)

print('Functions defined.')

Functions defined.


In [8]:
# CELL 8: TOPSIS fallback

NORM_COLS = ['Response_Time_norm', 'Availability_norm', 'Throughput_norm',
             'Reliability_norm', 'Latency_norm']

def run_topsis(group_df):
    group_df = group_df.sample(frac=1, random_state=int(group_df['list_id'].iloc[0])).reset_index(drop=True)
    w = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
    matrix      = group_df[NORM_COLS].values.astype(float)
    weighted    = matrix * w
    ideal_best  = weighted.max(axis=0)
    ideal_worst = weighted.min(axis=0)
    dist_best   = np.sqrt(((weighted - ideal_best)  ** 2).sum(axis=1))
    dist_worst  = np.sqrt(((weighted - ideal_worst) ** 2).sum(axis=1))
    closeness   = dist_worst / (dist_best + dist_worst + 1e-10)
    topsis_top1_idx = np.argmax(closeness)
    topsis_top1_did = group_df['display_id'].values[topsis_top1_idx]
    true_top1_did   = group_df[group_df['list_rank'] == 1]['display_id'].values[0]
    return int(topsis_top1_did == true_top1_did)

print('TOPSIS fallback defined.')

TOPSIS fallback defined.


In [9]:
# CELL 9: Run inference on VALIDATION set
print('Running inference on validation set...')
val_results = []
val_partial_path = RESULTS_PATH + 'val_confidence_results_partial.csv'

if os.path.exists(val_partial_path):
    done_df   = pd.read_csv(val_partial_path)
    done_ids  = set(done_df['list_id'].tolist())
    val_results = done_df.to_dict('records')
    print(f'Resuming val from checkpoint: {len(done_ids)} lists done')
else:
    done_ids = set()
    print('Starting val fresh')

val_list_ids = val_df['list_id'].unique().tolist()
remaining    = [lid for lid in val_list_ids if lid not in done_ids]
print(f'Val remaining: {len(remaining)} lists')

for i, lid in enumerate(remaining):
    if i % 10 == 0:
        print(f'  Val list {i}/{len(remaining)}...')
    group = val_df[val_df['list_id'] == lid].copy()
    generated_text, scores, group_shuffled = run_inference_with_logprobs(group)
    confidence = compute_confidence(scores, group_shuffled)
    pred_ranks = parse_ranking(generated_text, group_shuffled)
    top1       = compute_top1(group_shuffled, pred_ranks)
    val_results.append({
        'list_id':    lid,
        'confidence': confidence,
        'top1':       top1,
        'generated':  generated_text
    })
    if (i + 1) % 5 == 0:
        pd.DataFrame(val_results).to_csv(val_partial_path, index=False)
        print(f'  Val checkpoint saved: {len(val_results)} lists')

val_results_df = pd.DataFrame(val_results)
val_results_df.to_csv(RESULTS_PATH + 'val_confidence_results.csv', index=False)
print(f'Val done. Mean confidence: {val_results_df["confidence"].mean():.4f}')
print(f'Val Top-1: {val_results_df["top1"].mean()*100:.2f}%')

Running inference on validation set...
Starting val fresh
Val remaining: 251 lists
  Val list 0/251...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  margin=0.4071  entropy=0.9303  conf=0.6154
  margin=0.4215  entropy=0.9427  conf=0.6303
  margin=0.3658  entropy=0.8891  conf=0.5703
  margin=0.3720  entropy=0.9221  conf=0.5857
  margin=0.3734  entropy=0.9177  conf=0.5854
  Val checkpoint saved: 5 lists
  margin=0.4268  entropy=0.8958  conf=0.6183
  margin=0.3812  entropy=0.9337  conf=0.5966
  margin=0.3866  entropy=0.9141  conf=0.5945
  margin=0.3412  entropy=0.9116  conf=0.5577
  margin=0.3779  entropy=0.9077  conf=0.5857
  Val checkpoint saved: 10 lists
  Val list 10/251...
  margin=0.3738  entropy=0.9000  conf=0.5800
  margin=0.4005  entropy=0.9127  conf=0.6046
  margin=0.3708  entropy=0.9043  conf=0.5790
  margin=0.4271  entropy=0.9306  conf=0.6305
  margin=0.3893  entropy=0.9029  conf=0.5929
  Val checkpoint saved: 15 lists
  margin=0.4046  entropy=0.9272  conf=0.6125
  margin=0.3334  entropy=0.8553  conf=0.5340
  margin=0.4040  entropy=0.9111  conf=0.6067
  margin=0.4034  entropy=0.9345  conf=0.6140
  margin=0.3802  entropy=0

In [ ]:
# Load saved validation results
val_results_df = pd.read_csv(RESULTS_PATH + 'val_confidence_results.csv')
print(f'Loaded val results: {len(val_results_df)} lists')
print(f'Val Top-1: {val_results_df["top1"].mean()*100:.2f}%')
print(f'Val Mean confidence: {val_results_df["confidence"].mean():.4f}')

In [10]:
# CELL 10: Temperature scaling calibration on validation set
from scipy.optimize import minimize_scalar

raw_confidences = val_results_df['confidence'].values
true_labels     = val_results_df['top1'].values

def calibration_loss(temperature):
    eps = 1e-6
    log_odds = np.log(np.clip(raw_confidences, eps, 1-eps) /
                      (1 - np.clip(raw_confidences, eps, 1-eps)))
    scaled_probs = expit(log_odds / temperature)
    nll = -np.mean(
        true_labels * np.log(scaled_probs + eps) +
        (1 - true_labels) * np.log(1 - scaled_probs + eps)
    )
    return nll

result = minimize_scalar(calibration_loss, bounds=(0.1, 10.0), method='bounded')
OPTIMAL_TEMPERATURE = result.x
print(f'Optimal temperature: {OPTIMAL_TEMPERATURE:.4f}')

def apply_temperature_scaling(confidence, temperature):
    eps = 1e-6
    log_odds = np.log(np.clip(confidence, eps, 1-eps) /
                      (1 - np.clip(confidence, eps, 1-eps)))
    return float(expit(log_odds / temperature))

print('Temperature scaling calibrated.')

Optimal temperature: 0.1689
Temperature scaling calibrated.


In [11]:
# CELL 11: Run inference on TEST set
partial_path = RESULTS_PATH + 'test_confidence_results_partial.csv'
full_path    = RESULTS_PATH + 'test_confidence_results.csv'

done_ids     = set()
test_results = []

if os.path.exists(partial_path):
    done_df      = pd.read_csv(partial_path)
    done_ids     = set(done_df['list_id'].tolist())
    test_results = done_df.to_dict('records')
    print(f'Resuming from checkpoint: {len(done_ids)} lists already done')

test_list_ids = test_df['list_id'].unique().tolist()
remaining     = [lid for lid in test_list_ids if lid not in done_ids]
print(f'Remaining: {len(remaining)} lists to process')

for i, lid in enumerate(remaining):
    if i % 10 == 0:
        print(f'  List {i}/{len(remaining)}...')

    base_group = test_df[test_df['list_id'] == lid].copy().reset_index(drop=True)

    llm_group = base_group.copy()
    generated_text, scores, group_shuffled = run_inference_with_logprobs(llm_group)
    raw_confidence = compute_confidence(scores, group_shuffled)
    cal_confidence = apply_temperature_scaling(raw_confidence, OPTIMAL_TEMPERATURE)
    pred_ranks     = parse_ranking(generated_text, group_shuffled)
    llm_top1       = compute_top1(group_shuffled, pred_ranks)

    topsis_group = base_group.copy()
    topsis_top1  = run_topsis(topsis_group)

    test_results.append({
        'list_id':        lid,
        'raw_confidence': raw_confidence,
        'cal_confidence': cal_confidence,
        'llm_top1':       llm_top1,
        'topsis_top1':    topsis_top1,
        'generated':      generated_text
    })

    if (i + 1) % 5 == 0:
        pd.DataFrame(test_results).to_csv(partial_path, index=False)
        print(f'  Checkpoint saved: {len(test_results)} lists total')

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(full_path, index=False)
print(f'\nDone! {len(test_results_df)} lists processed')
print(f'LLM Top-1:    {test_results_df["llm_top1"].mean()*100:.2f}%')
print(f'TOPSIS Top-1: {test_results_df["topsis_top1"].mean()*100:.2f}%')
print(f'\nCalibrated confidence stats:')
print(test_results_df['cal_confidence'].describe().round(4))
print(f'\nLists below each threshold:')
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    below = (test_results_df['cal_confidence'] < t).sum()
    print(f'  Below {t}: {below} lists ({below/len(test_results_df)*100:.1f}%)')

Remaining: 251 lists to process
  List 0/251...
  margin=0.4060  entropy=0.9185  conf=0.6107
  margin=0.3787  entropy=0.9108  conf=0.5873
  margin=0.4028  entropy=0.9052  conf=0.6038
  margin=0.3874  entropy=0.9070  conf=0.5927
  margin=0.3604  entropy=0.8953  conf=0.5680
  Checkpoint saved: 5 lists total
  margin=0.3051  entropy=0.8602  conf=0.5123
  margin=0.3770  entropy=0.9022  conf=0.5832
  margin=0.3658  entropy=0.9270  conf=0.5823
  margin=0.3882  entropy=0.9236  conf=0.5988
  margin=0.3522  entropy=0.9091  conf=0.5658
  Checkpoint saved: 10 lists total
  List 10/251...
  margin=0.3795  entropy=0.9001  conf=0.5844
  margin=0.3868  entropy=0.9335  conf=0.6009
  margin=0.3670  entropy=0.8984  conf=0.5742
  margin=0.3833  entropy=0.9166  conf=0.5927
  margin=0.3603  entropy=0.9244  conf=0.5771
  Checkpoint saved: 15 lists total
  margin=0.3920  entropy=0.9138  conf=0.5985
  margin=0.3923  entropy=0.9428  conf=0.6082
  margin=0.4083  entropy=0.8974  conf=0.6053
  margin=0.4024  entr

In [12]:
# CELL 12: Confidence-gated selection
THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9]
gate_results = []

print('Confidence-gated selection results:')
print(f'{"T":<6} {"Coverage(%)":<15} {"Accepted(n)":<15} {"Accuracy(%)":<15} {"Fallback(n)":<15} {"Combined(%)":<15}')
print('='*80)

for T in THRESHOLDS:
    accepted = test_results_df[test_results_df['cal_confidence'] >= T]
    rejected = test_results_df[test_results_df['cal_confidence'] <  T]

    n_accepted  = len(accepted)
    n_rejected  = len(rejected)
    n_total     = len(test_results_df)
    coverage    = round(n_accepted / n_total * 100, 2)
    acc_accepted = round(accepted['llm_top1'].mean() * 100, 2) if n_accepted > 0 else 0.0
    correct_accepted = accepted['llm_top1'].sum()
    correct_rejected = rejected['topsis_top1'].sum()
    combined_acc = round((correct_accepted + correct_rejected) / n_total * 100, 2)

    print(f'{T:<6} {coverage:<15} {n_accepted:<15} {acc_accepted:<15} {n_rejected:<15} {combined_acc:<15}')
    gate_results.append({
        'T': T, 'coverage': coverage, 'n_accepted': n_accepted,
        'n_rejected': n_rejected, 'acc_accepted': acc_accepted,
        'combined_acc': combined_acc
    })

gate_df = pd.DataFrame(gate_results)
gate_df.to_csv(RESULTS_PATH + 'confidence_gating_results.csv', index=False)
print('\nSaved: confidence_gating_results.csv')

Confidence-gated selection results:
T      Coverage(%)     Accepted(n)     Accuracy(%)     Fallback(n)     Combined(%)    
0.5    100.0           251             93.63           0               93.63          
0.6    99.6            250             94.0            1               93.63          
0.7    98.8            248             93.95           3               93.63          
0.8    95.62           240             94.58           11              93.23          
0.9    60.16           151             96.03           100             92.43          

Saved: confidence_gating_results.csv


In [15]:
correct = test_results_df[test_results_df['llm_top1'] == 1]['cal_confidence']
wrong   = test_results_df[test_results_df['llm_top1'] == 0]['cal_confidence']

print(f'Correct — mean: {correct.mean():.4f}  std: {correct.std():.4f}  min: {correct.min():.4f}  max: {correct.max():.4f}')
print(f'Wrong   — mean: {wrong.mean():.4f}  std: {wrong.std():.4f}  min: {wrong.min():.4f}  max: {wrong.max():.4f}')

Correct — mean: 0.9028  std: 0.0483  min: 0.6073  max: 0.9773
Wrong   — mean: 0.8543  std: 0.0933  min: 0.5723  max: 0.9504


In [14]:
# CELL 14: Summary

print(f'Optimal temperature:       {OPTIMAL_TEMPERATURE:.4f}')
print(f'LLM baseline Top-1:        {test_results_df["llm_top1"].mean()*100:.2f}%')
print(f'TOPSIS baseline Top-1:     {test_results_df["topsis_top1"].mean()*100:.2f}%')
print()
print('Confidence-gated selection:')
print(gate_df.to_string(index=False))
print()
print('Files saved to:', RESULTS_PATH)

Optimal temperature:       0.1689
LLM baseline Top-1:        93.63%
TOPSIS baseline Top-1:     91.63%

Confidence-gated selection:
  T  coverage  n_accepted  n_rejected  acc_accepted  combined_acc
0.5    100.00         251           0         93.63         93.63
0.6     99.60         250           1         94.00         93.63
0.7     98.80         248           3         93.95         93.63
0.8     95.62         240          11         94.58         93.23
0.9     60.16         151         100         96.03         92.43

Files saved to: /content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_ Calibrated and Adaptive Listwise Listwise Ranking/Llama3.1-8B/
